# ZMART target acquisition

Run the cells from top to bottom. Edit the setup values, select the requested LAS X jobs when asked, and inspect the plots before acquiring targets.

## 1 · Setup and connect
Edit the analysis repo, then run.

In [ ]:
import sys
from pathlib import Path as _Path
from IPython import get_ipython

_shell = get_ipython()
if _shell is not None:
    _shell.run_line_magic("matplotlib", "widget")

_target_acq = _Path("workflows/target_acquisition")
if not _target_acq.is_dir():
    _target_acq = _Path.cwd()
sys.path.insert(0, str(_target_acq.resolve()))
from _bootstrap import Path, workflow

ANALYSIS_REPO = Path("C:/code/smart-analysis")
SIMULATE_IMAGES = False
AUTOFOCUS_JOB = None  # Set only when LAS X has more than one autofocus job.

if "engine" in globals():
    try:
        engine.shutdown()
    except Exception as exc:
        print(f"warning: the previous analysis engine did not shut down cleanly: {exc}")
    del engine
if "zmart_controller" in globals():
    try:
        zmart_controller.disconnect()
    except Exception as exc:
        print(f"warning: the previous session did not disconnect cleanly: {exc}")
    del zmart_controller

engine = workflow.load_analysis_engine(ANALYSIS_REPO)
try:
    workflow.preflight_analysis_engine(engine)
    zmart_controller = workflow.connect("leica")
    ROOT = Path(zmart_controller.run_procedure({"name": "get_root"})["root"])
except Exception:
    try:
        if "zmart_controller" in globals():
            zmart_controller.disconnect()
            del zmart_controller
    finally:
        engine.shutdown()
        del engine
    raise
ROOT

### Where the run stands

A one-glance checklist built from the notebook's own variables — re-run this cell after any step to see what is done, what is still to do, and what deserves a second look. It never talks to the microscope.

In [ ]:
workflow.print_run_status(globals())

## 2 · Set origin
Move to the run origin first. This makes the current position `(0, 0, 0)`.

In [ ]:
zmart_controller.set_origin()

## 3a · Overview job

Select the low-magnification overview job in LAS X, then run.

In [ ]:
overview_state = zmart_controller.get_state()
limits = overview_state["observed"].get("limits")
if not limits or limits.get("is_fallback") or limits.get("source") != "machine":
    raise RuntimeError(
        f"machine-specific stage limits are not active (got {limits}); publish this "
        "machine's measured envelope with limits/notebooks/set_stage_limits.ipynb first"
    )
overview_state["observed"]

## 3b · Target job

Select the high-magnification target job in LAS X, then run.

In [ ]:
target_state = zmart_controller.get_state()
if target_state["changeable"].get("job") == overview_state["changeable"].get("job"):
    raise RuntimeError("the overview and target jobs are the same; select the high-magnification target job")
target_state["observed"]

## 4 · Initial positions
Read overview tile centres from the controller.

In [ ]:
zmart_controller.set_state(overview_state)
positions = zmart_controller.run_procedure({"name": "get_positions"})["positions"]
print(len(positions), "overview positions")

## 5 · Focus

Pick focus points on the map, then press **Measure focus**. The microscope visits each point, runs the autofocus job there, and the fitted focus map appears as a heatmap in the same figure.

- **Left-click** adds a focus point; **right-click** removes the nearest one.
- Points already placed in LAS X are pre-filled.
- The setup cell activates the interactive notebook backend automatically.

Re-measuring after editing points only visits the new or moved points — everything already measured this session is reused. Once measured, the overview tile markers take the colour of the fitted focus map.

In [ ]:
picker = workflow.pick_focus_points(
    zmart_controller, positions, af_job=AUTOFOCUS_JOB
)

In [ ]:
focus = picker.require_focus()
workflow.plot_focus_surface(focus, save_path=ROOT / "focus_surface.png", show=True)
print("focus model:", focus.model)

## 6 · Overview

Capture one overview at each position and watch the map grow: every tile appears at its real stage position the moment it is saved, while the microscope scans the rest. Pan and zoom with the toolbar. The controls on the right work per channel: pick a channel, toggle **show**, step its **colour**, and drag the **display range** slider for brightness and contrast (the same min/max control as Fiji's B&C).

In [ ]:
viewer = workflow.view_overview()
viewer.expect_tiles(len(positions))  # lets the map show progress + time left
overview_records = workflow.run_overview(
    zmart_controller, positions, state=overview_state, focus=focus,
    on_record=viewer.add_acquisition,
)
print(len(overview_records), "overviews captured")
if SIMULATE_IMAGES:
    n = workflow.hijack_if_simulating(overview_records, simulate=True)
    viewer.reload()  # the hijack rewrote the saved pixels
    print(n, "overview images replaced")

In [ ]:
overviews = workflow.overview_inputs_from_records(positions, overview_records, focus=focus)

## 7 · Discover targets

Segment the overview images, then decide which cells become targets in the explorer:

- the radio lists put any measured feature on the plot's x and y axes (position, area, brightness, ...);
- gate with the two range sliders (thresholds on the current axes) and/or by drawing a lasso around points with the mouse — gated points stay blue, the rest fade out;
- hover near a point to see that cell's image crop in the side panel.

`explorer.gated` is the list the next step samples from.

The same cells also appear as rings on the overview map above — blue when they pass the gate, grey when they do not. After editing the gate, call `viewer.refresh_targets()` to recolour them.

**Picking specific cells**: double-click a dot to pick that exact cell (double-click again to un-pick; a single drag still draws the lasso). Picked cells wear a black outline; `gallery.acquire_selected()` in the next step images exactly them. Cells already acquired this session turn green on the scatter and the map, so nothing gets imaged twice by accident.

In [ ]:
targets = workflow.discover_targets(engine, overviews)
print(len(targets), "targets discovered")

In [ ]:
explorer = workflow.explore_targets(targets, overviews)
# The same cells on the overview map (call viewer.refresh_targets() after
# editing the gate to recolour).
viewer.show_targets(targets, explorer)

## 8 · Acquire targets

Type how many targets to acquire and press **Acquire**. That many cells are drawn at random from the gate, re-imaged at the target job, and shown below as pairs — the overview crop (left) and the new high-magnification image (right), both over the same physical window so the cell appears at the same scale in both.

After the run you can record your judgement of each pair in code — `gallery.set_verdict(0, "good")` / `"bad"` — and write the record into the run folder with `gallery.save_curation(ROOT)`. (The React edition has ✓/✗ buttons on each row for the same thing.)

`gallery.acquire_selected()` images exactly the cells you picked in the explorer (each pick is re-checked against the gate first); the **Acquire** button still draws a random sample. While the run works, the figure's status line counts pairs and estimates the time left.

In [ ]:
gallery = workflow.acquire_gallery(
    zmart_controller,
    explorer,
    overviews,
    state=target_state,
    focus=focus,
    after_acquire=lambda records: workflow.hijack_if_simulating(records, simulate=SIMULATE_IMAGES),
)

## 9 · Summary
Write the summary and run map.

In [ ]:
if not gallery.records:
    raise RuntimeError("no targets have been acquired; use the Acquire button first")

summary = workflow.write_run_report(
    ROOT,
    positions=positions,
    focus=focus,
    overview_records=overview_records,
    targets=gallery.picked,
)
summary

In [ ]:
try:
    if "engine" in globals():
        engine.shutdown()
        del engine
finally:
    zmart_controller.disconnect()
    del zmart_controller